#### trial for multiple employers and cities

In [ ]:
import os
import time
import logging
import traceback
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.action_chains import ActionChains

# Setup logging
logging.basicConfig(
    level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S',
    force=True)  # Force logging to overwrite any previous settings

# Load credentials from environment variables (recommended)
EMAIL = "bmorris@lacydiversified.com"
PASSWORD = "Oiaglobal25!"

employer_city_list = [{"Employer Name": "Polygon Co", "City": "Walkerton"},
                      {"Employer Name": "Acme Barricades", "City": "Jacksonville"},
                      {"Employer Name": "Asian Buffet", "City": "Linton"},
                      {"Employer Name": "thatolife Llp", "City": "Indianapolis"}]

# Start browser
driver = webdriver.Chrome()
wait = WebDriverWait(driver, 20)
results = []

# Step 1: Login
driver.get("https://www.capitaliq.spglobal.com")
email_input = wait.until(EC.presence_of_element_located((By.ID, "input28")))
email_input.clear()
email_input.send_keys(EMAIL)
next_button = wait.until(EC.element_to_be_clickable((By.XPATH, '//input[@type="submit" and @value="Next"]')))
next_button.click()
logging.info("Email submitted.")
password_input = wait.until(EC.presence_of_element_located((By.XPATH, '//input[@placeholder="Password"]')))
password_input.send_keys(PASSWORD)
wait.until(EC.element_to_be_clickable((By.XPATH, '//input[@type="submit" and @value="Sign In"]'))).click()
logging.info("Password submitted and logged in.")
wait.until(EC.presence_of_element_located((By.XPATH, '//div[contains(@class, "header")]')))
logging.info("Login successful and page loaded.")

# Step 2: Accept cookie popup if present
try:
    cookie_button = WebDriverWait(driver, 5).until(
        EC.element_to_be_clickable((By.XPATH, '//button[contains(text(), "Accept") or contains(text(), "Agree")]')))
    cookie_button.click()
    logging.info("Cookie popup accepted.")
except:
    logging.info("No cookie popup found or already handled.")

# Step 3: Loop through employers
for entry in employer_city_list:
    search_term = entry["Employer Name"]
    search_city = entry["City"]
    search_url = f"https://www.capitaliq.spglobal.com/apisv3/spg-webplatform-core/search/searchResults?q={search_term.replace(' ', '%20')}&vertical=private_companies_mi-gss"
    driver.get(search_url)
    logging.info(f"Navigated to search results for: {search_term}")

    try:
        # Wait for either results label or "no results" icon to appear
        wait.until(
            EC.any_of(
                EC.presence_of_element_located((By.CLASS_NAME, "css-7mog2u")),
                EC.presence_of_element_located((By.XPATH, "//*[@data-testid='icon' and contains(@class, 'css-5pnr6z')]"))))
    except Exception as e:
        logging.error(f"Timeout waiting for search results or no-results icon for {search_term}: {e}")
        results.append({"Employer Name": search_term, "City": search_city, "Match Found": False})
        continue

    # Accept cookie popup if present
    try:
        cookie_button = WebDriverWait(driver, 10).until(
            EC.element_to_be_clickable((By.XPATH, '//button[contains(text(), "Accept") or contains(text(), "Agree")]')))
        cookie_button.click()
        logging.info("Cookie popup accepted.")
    except Exception:
        logging.info("No cookie popup found or already handled.")

    # Change results per page to 50

    # Check if no data was returned from the search
    try:
        no_results_element = driver.find_element(By.XPATH,"//*[@data-testid='icon' and contains(@class, 'css-5pnr6z')]")
        if no_results_element.is_displayed():
            results.append({"Employer Name": search_term, "City": search_city, "Match Found": False})
            logging.info(f"No search results for {search_term} in {search_city}.")
            continue
    except Exception:
        pass
    try:
        show_label = wait.until(EC.presence_of_element_located((By.CLASS_NAME, "css-7mog2u")))
        driver.execute_script("arguments[0].scrollIntoView(true);", show_label)
        time.sleep(5)
        dropdown_trigger = wait.until(EC.element_to_be_clickable((By.XPATH, '//button[@type="button" and @title="20"]')))
        driver.execute_script("arguments[0].click();", dropdown_trigger)
        time.sleep(5)
        options = wait.until(EC.presence_of_all_elements_located((By.XPATH, '//div[@title="50" and contains(@class, "css-11d71qm")]')))
        for option in options:
            if option.is_displayed():
                driver.execute_script("arguments[0].click();", option)
                time.sleep(3)
                break
    except Exception as e:
        logging.warning(f"Dropdown error: {e}")

    # Search for match
    match_found = False
    try:
        WebDriverWait(driver, 10).until(
            EC.presence_of_all_elements_located((By.XPATH, '//a[contains(@href, "/web/client#company/profile?id=")]')))
        company_links = driver.find_elements(By.XPATH, '//a[contains(@href, "/web/client#company/profile?id=")]')
        for link in company_links:
            try:
                container = link.find_element(By.XPATH, './ancestor::div[3]')
                if search_city.lower() in container.text.lower():
                    match_found = True
                    # results.append({"Employer Name": search_term, "City": search_city, "Match Found": True})
                    driver.execute_script("arguments[0].click();", link)
                    company_data = {
                        "NAICS Code": None,
                        "Industry": None,
                        "Description": None,
                        "Website URL": None,
                        "Headquarters": None,
                        "Current_Headcount": None,
                        "Percent_Change_in_HC": None,
                        "Change_in_HC": None,
                        "CEO Name": None,
                        "CEO Profile Link": None,
                        "Key People": None
                    }

                    # 1. NAICS Code
                    try:
                        naics_element = wait.until(EC.presence_of_element_located(
                            (By.XPATH, '//div[@id="naics"]//span[@id="naicsLessTextSpan"]/b')))
                        company_data["NAICS Code"] = naics_element.text.strip()
                    except Exception as e:
                        print("NAICS Code not found:", e)

                    # 2. Industry
                    try:
                        industry_element = wait.until(EC.presence_of_element_located(
                            (By.CSS_SELECTOR, 'h6[data-testid="company-industry-mini-body"]')))
                        company_data["Industry"] = industry_element.text.strip()
                    except Exception as e:
                        print("Industry not found!")

                    # 3. Description
                    try:
                        view_all_button = wait.until(EC.element_to_be_clickable(
                            (By.XPATH, '//a[contains(text(), "VIEW ALL")]')))
                        view_all_button.click()
                        time.sleep(1)
                        description_element = wait.until(EC.presence_of_element_located((By.ID, "compDesc")))
                        description = description_element.text.strip()
                        if "VIEW LESS" in description:
                            description = description.split("VIEW LESS")[0].strip()
                        company_data["Description"] = description
                    except Exception as e:
                        print("Description not found!")

                    # 4. Website URL
                    try:
                        cpweblink_element = wait.until(EC.presence_of_element_located((By.CLASS_NAME, "cpweblink")))
                        company_data["Website URL"] = cpweblink_element.find_element(By.TAG_NAME, "a").get_attribute("href")
                    except Exception as e:
                        print("Website URL not found!")

                    # 5. Headquarters
                    try:
                        block_label_element = wait.until(EC.presence_of_element_located((By.CLASS_NAME, "blockLabel")))
                        hq_element = block_label_element.find_element(By.XPATH, "following-sibling::*[1]")
                        lines = hq_element.text.splitlines()
                        company_data["Headquarters"] = ', '.join([line.strip() for line in lines if line.strip()])
                    except Exception as e:
                        print("Headquarters not found!")

                    # 6. Current Headcount
                    try:
                        headcount_element = wait.until(EC.presence_of_element_located((
                            By.XPATH, '//div[@class="spg-d-flex"]/h6[@class="spg-heading spg-heading--xxsmall spg-text-nowrap"]')))
                        company_data["Current_Headcount"] = headcount_element.text.strip()
                    except Exception as e:
                        print("Current Headcount not found!")

                    # 7. Percent Change in Headcount
                    try:
                        percent_change = wait.until(EC.presence_of_element_located((By.XPATH, '//strong[@class="spg-ml-xs"]')))
                        company_data["Percent_Change_in_HC"] = percent_change.text.strip().replace('%', '')
                    except Exception as e:
                        print("Percent Change in Headcount not found!")

                    # 8. Change in Headcount Direction
                    try:
                        change_type = wait.until(EC.presence_of_element_located((By.XPATH,
                            '//div[contains(@class, "spg-d-flex") and contains(@class, "spg-align-center")]//span[@data-icon]')))
                        icon_html = change_type.get_attribute('outerHTML')
                        if 'caret-up' in icon_html:
                            company_data["Change_in_HC"] = 1
                        elif 'caret-down' in icon_html:
                            company_data["Change_in_HC"] = -1
                        else:
                            company_data["Change_in_HC"] = 0
                    except Exception as e:
                        print("Change in Headcount direction not found!")

                    # 9. CEO Name and Profile Link
                    try:
                        ceo_element = wait.until(EC.presence_of_element_located((
                            By.CSS_SELECTOR, 'a[data-testid="ribbon-ceo"].spg-text-medium.css-12tcig2.css-1b9i013')))
                        company_data["CEO Name"] = ceo_element.text
                        company_data["CEO Profile Link"] = ceo_element.get_attribute("href")
                    except Exception as e:
                        print("CEO information not found!")

                    # 10. Officers
                    # Click the "More" link under Officers & Directors
                    more_button = wait.until(EC.element_to_be_clickable((By.XPATH, '//div[@id="offDirectors"]//a[contains(text(), "More")]')))
                    driver.execute_script("arguments[0].click();", more_button)
                    logging.info("Clicked on 'More' under Officers & Directors.")
                    time.sleep(5)

                    # Step 2: Wait for the table to appear
                    try:
                        wait.until(EC.presence_of_element_located((By.ID, "section_1_control_23_section_0_control_71_i2_grid_table")))
                        table = driver.find_element(By.ID, 'section_1_control_23_section_0_control_71_i2_grid_table')
                    except Exception as e:
                        logging.error(f"Table not found: {e}")
                        driver.quit()
                        exit()

                    # Step 3: Loop through rows and print all <td> contents
                    rows = table.find_elements(By.XPATH, './/tbody/tr')
                    merged_data = []
                    for i, row in enumerate(rows):
                        try:
                            name = row.find_element(By.XPATH, './/td[@data-colkey="_NameFromDoc"]').text.strip()
                            title = row.find_element(By.XPATH, './/td[@data-colkey="_Title"]/div').text.strip()
                            merged_data.append(f" {title} : {name if name else '[No Name]'}")
                        except Exception as e:
                            logging.warning(f"Could not process row {i+1}: {e}")
                    final_output = '; '.join(merged_data)
                    company_data["Key People"] = final_output
                    combined_result = {
                        "Employer Name": search_term,
                        "City": search_city,
                        "Match Found": True
                    }
                    combined_result.update(company_data)
                    results.append(combined_result)
                    logging.info(f"Data collected for {search_term} in {search_city}.")
                    break
            except:
                continue
        if not match_found:
            results.append({"Employer Name": search_term, "City": search_city, "Match Found": False})
    except Exception as e:
        logging.error(f"Search error for {search_term}: {e}")
        results.append({"Employer Name": search_term, "City": search_city, "Match Found": False})
    

# Wrap up
driver.quit()

df = pd.DataFrame(results)
df

2025-07-15 17:08:38 - INFO - Email submitted.
2025-07-15 17:08:39 - INFO - Password submitted and logged in.
2025-07-15 17:08:39 - INFO - Login successful and page loaded.
2025-07-15 17:08:44 - INFO - No cookie popup found or already handled.
2025-07-15 17:08:45 - INFO - Navigated to search results for: Polygon Co
2025-07-15 17:08:49 - INFO - Cookie popup accepted.
2025-07-15 17:09:06 - INFO - Clicked on 'More' under Officers & Directors.
2025-07-15 17:09:11 - INFO - Data collected for Polygon Co in Walkerton.
2025-07-15 17:09:12 - INFO - Navigated to search results for: Acme Barricades
2025-07-15 17:09:22 - INFO - No cookie popup found or already handled.


CEO information not found!


2025-07-15 17:10:20 - INFO - Navigated to search results for: Asian Buffet
2025-07-15 17:10:31 - INFO - No cookie popup found or already handled.
2025-07-15 17:10:45 - INFO - Navigated to search results for: thatolife Llp
2025-07-15 17:10:55 - INFO - No cookie popup found or already handled.
2025-07-15 17:10:55 - INFO - No search results for thatolife Llp in Indianapolis.


,Employer Name,City,Match Found,NAICS Code,Industry,Description,Website URL,Headquarters,Current_Headcount,Percent_Change_in_HC,Change_in_HC,CEO Name,CEO Profile Link,Key People
0,Polygon Co,Walkerton,True,326121 - Unlaminated Plastics Profile Shape Ma...,Commodity Chemicals,"Polygon Company, Inc., an engineered materials...",http://www.polygoncomposites.com/,"103 Industrial Park Drive, PO Box 176, Walkert...",71,8,-1.0,Bree Katulak,https://www.capitaliq.spglobal.com/web/client#...,President & CEO : Bree Katulak; Vice Preside...
1,Asian Buffet,Linton,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,thatolife Llp,Indianapolis,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
results

[{'Employer Name': 'Polygon Co', 'City': 'Walkerton', 'Match Found': True},
 {'Employer Name': 'Acme Barricades',
  'City': 'Jacksonville',
  'Match Found': True},
 {'Employer Name': 'Asian Buffet', 'City': 'Linton', 'Match Found': False},
 {'Employer Name': 'thatolife Llp',
  'City': 'Indianapolis',
  'Match Found': False}]